# IBM Experiments — Reproduce Paper Results

This notebook is a clean reproduction notebook for the IBM dataset. It keeps the main implementation in `src/ibm_pipeline.py` and only calls the pipeline functions here.

Expected input: the IBM transactions dataset downloaded locally and placed under `data/`. Update `IBM_DATA_PATH` below before running.

In [ ]:
from pathlib import Path
import sys

# Make repository root importable when this notebook is run from notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.ibm_pipeline import (
    IBMFraudConfig,
    load_and_preprocess,
    run_pipeline_mlp,
    run_pipeline,
    run_all_strategies,
    run_gcn_comparison,
    run_sage_comparison,
)

PROJECT_ROOT

## 1. Configuration

For a fast smoke test, reduce `MAX_EPOCHS` and `N_SPLITS`. For paper reproduction, keep the default values.

In [ ]:
cfg = IBMFraudConfig()

# Uncomment for a quick test run only
# cfg.MAX_EPOCHS = 5
# cfg.N_SPLITS = 2
# cfg.PATIENCE_CHECKS = 2

IBM_DATA_PATH = PROJECT_ROOT / "data" / "ibm_transactions.csv"
IBM_DATA_PATH

## 2. Load and preprocess IBM data

In [ ]:
df = load_and_preprocess(str(IBM_DATA_PATH), cfg)
df.head()

## 3. Run the no-graph MLP baseline

This is the empirical anchor: same features, splits, threshold tuning, and evaluation protocol, but no graph.

In [ ]:
mlp_results = run_pipeline_mlp(df, cfg)
mlp_results

## 4. Run the three graph topology strategies with GATv2

These are the core topology-isolation experiments: `multi_relation`, `hybrid`, and `intra_group`.

In [ ]:
gatv2_results = run_all_strategies(df, cfg)
gatv2_results

## 5. Optional architecture comparison

Run GCN and GraphSAGE comparisons only when you need the architecture-sensitivity tables.

In [ ]:
# gcn_results = run_gcn_comparison(df, cfg)
# sage_results = run_sage_comparison(df, cfg)

## 6. Save results

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# If the pipeline returns pandas DataFrames, these will save directly.
# Otherwise, convert results manually after inspecting the returned objects.
try:
    gatv2_results.to_csv(RESULTS_DIR / "ibm_gatv2_results.csv", index=False)
except AttributeError:
    print("gatv2_results is not a DataFrame; inspect it before saving.")

try:
    mlp_results.to_csv(RESULTS_DIR / "ibm_mlp_baseline.csv", index=False)
except AttributeError:
    print("mlp_results is not a DataFrame; inspect it before saving.")